In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE       = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
HORAS_IN   = BASE / "data/processed/horas_por_setor_2023_corrigido.csv"
PIB_IN     = BASE / "data/processed/pib_setorial_2012_2023.csv"
BASE_OUT   = BASE / "data/processed/base_analitica_setorial_2023.csv"
TAB_OUT    = BASE / "outputs/tables/impacto_setorial_hipotese_nula.csv"
DIR_FIGS   = BASE / "outputs/figures"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Imports OK")

In [ ]:
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

pib_2023     = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6

print("Colunas horas:")
print(df_horas.columns.tolist())
print(f"\nPIB total 2023: R$ {pib_total_rs/1e12:.2f} tri")

print("\n── pct_41_44h (corrigido — exclui 40h exatas) ───────────")
print(df_horas[["n", "media", "pct_abaixo_40", "pct_40h_exatas",
                "pct_41_44h", "pct_acima_44h"]].to_string())

In [ ]:
correspondencia = [
    ("Agropecuária",          "agropecuaria",          True),
    ("Indústria geral",       "ind_transformacao",      True),
    ("Construção",            "construcao",             True),
    ("Comércio e rep.",       "comercio",               True),
    ("Transp. e armaz.",      "transporte",             True),
    ("Inf. e serv. prof.",    "info_comunicacao",       True),
    ("Adm. públ. e educação", "adm_publica_saude_educ", True),
    ("Alojamento e alim.",    "outros_servicos",        False),
    ("Saúde",                 "adm_publica_saude_educ", False),
    ("Serv. domésticos",      "outros_servicos",        False),
]

In [ ]:
SEMANAS_ANO    = 52
H_MEDIA_AFETADOS = 42.0   
H_PROPOSTA       = 40.0
REDUCAO_AFETADOS = (H_PROPOSTA - H_MEDIA_AFETADOS) / H_MEDIA_AFETADOS  # -4.76%

print(f"Redução assumida para trabalhadores na faixa 41-44h: {REDUCAO_AFETADOS*100:.2f}%")
print(f"(de {H_MEDIA_AFETADOS:.0f}h para {H_PROPOSTA:.0f}h)\n")

registros = []

for setor, col, incluir in correspondencia:
    if setor not in df_horas.index:
        print(f"⚠ Não encontrado: {setor}")
        continue

    n_ocupados   = df_horas.loc[setor, "n"]
    h_media      = df_horas.loc[setor, "media"]
    h_mediana    = df_horas.loc[setor, "mediana"]

    # USANDO COLUNA CORRIGIDA: pct_41_44h (exclui 40h exatas)
    pct_41_44    = df_horas.loc[setor, "pct_41_44h"]
    pct_acima_44 = df_horas.loc[setor, "pct_acima_44h"]

    vab_rs = pib_2023[col] * 1e6

    # Total de horas/ano no setor
    total_horas_ano = n_ocupados * h_media * SEMANAS_ANO

    # Produtividade por hora
    produt_hora = vab_rs / total_horas_ano

    # Parcela afetada = apenas trabalhadores na faixa 41-44h
    parcela_afetada   = pct_41_44 / 100.0
    delta_horas_setor = parcela_afetada * REDUCAO_AFETADOS
    delta_vab_rs      = vab_rs * delta_horas_setor

    registros.append({
        "setor":                setor,
        "col_cnt":              col,
        "incluir":              incluir,
        "n_ocupados":           int(n_ocupados),
        "h_media":              round(h_media, 2),
        "pct_41_44h":           round(pct_41_44, 2),   # CORRIGIDO
        "pct_acima_44h":        round(pct_acima_44, 2),
        "vab_r$_bi":            round(vab_rs / 1e9, 2),
        "produt_r$_hora":       round(produt_hora, 2),
        "parcela_afetada":      round(parcela_afetada, 4),
        "delta_horas_setor":    round(delta_horas_setor * 100, 4),
        "delta_vab_r$_bi":      round(delta_vab_rs / 1e9, 3),
        "delta_vab_pct_setor":  round(delta_horas_setor * 100, 3),
    })

df_base = pd.DataFrame(registros)

print("── Base analítica setorial 2023 (CORRIGIDA) ─────────────")
cols = ["setor", "h_media", "pct_41_44h", "vab_r$_bi", "produt_r$_hora",
        "delta_vab_r$_bi", "delta_vab_pct_setor"]
print(df_base[cols].to_string(index=False))

In [ ]:
df_sim = df_base[df_base["incluir"] == True].copy()

delta_vab_total  = df_sim["delta_vab_r$_bi"].sum() * 1e9
delta_pib_pct    = (delta_vab_total / pib_total_rs) * 100
delta_pib_bi     = delta_vab_total / 1e9

print("══ RESULTADO — HIPÓTESE NULA (sem ganho de produtividade) ══")
print(f"Setores incluídos: {df_sim['setor'].tolist()}")
print(f"\nΔVAB total:              R$ {delta_pib_bi:.2f} bi")
print(f"ΔPIB estimado:           {delta_pib_pct:.3f}%")
print(f"\nReferência CNI:          -0,70%")
print(f"Nosso modelo:            {delta_pib_pct:.3f}%")
print(f"\nDetalhamento por setor:")
print(df_sim[["setor", "pct_41_44h", "delta_vab_r$_bi", "delta_vab_pct_setor"]]
      .sort_values("delta_vab_r$_bi").to_string(index=False))

In [ ]:
df_base.to_csv(BASE_OUT, index=False)
df_sim[cols].to_csv(TAB_OUT, index=False)
print(f"\n✓ Base salva: {BASE_OUT}")
print(f"✓ Tabela salva: {TAB_OUT}")

In [ ]:
df_g = df_sim.sort_values("delta_vab_r$_bi", ascending=True)

fig, ax = plt.subplots(figsize=(15, 8))

cores = ["#E84545" if v < -2 else "#F5A623" if v < -1 else "#2C6FBF"
         for v in df_g["delta_vab_r$_bi"]]

bars = ax.barh(df_g["setor"], df_g["delta_vab_r$_bi"], color=cores, height=0.6)
ax.axvline(0, color="black", linewidth=0.8)

for bar, (_, row) in zip(bars, df_g.iterrows()):
    val = row["delta_vab_r$_bi"]
    ax.text(
        val - 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"R$ {val:.2f} bi  ({row['delta_vab_pct_setor']:.3f}%)",
        va="center", ha="right", fontsize=5
    )

ax.set_xlabel("Variação estimada no VAB (R$ bilhões)", fontsize=9)
ax.set_title(
    "Impacto setorial da redução de jornada (44h→40h) — hipótese nula\n"
    "Sem ganho de produtividade · Trabalhadores na faixa 41–44h (excluindo 40h exatas) · Brasil 2023",
    fontsize=12, loc="left"
)

fig.text(
    0.01, -0.03,
    f"ΔPIB estimado: {delta_pib_pct:.3f}% (R$ {delta_pib_bi:.1f} bi) · "
    "Referência CNI: -0,70% · "
    "Faixa afetada: 41–44h, redução média de 42h→40h (-4,76%)",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "11_impacto_setorial_hipotese_nula.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 11 salvo.")